#### Model with Gating at position G (at attention calculation step) :  As in Paper where stated best effective

In [1]:
### Aplied gated atention at position G elementwise

In [2]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [3]:
import random
import numpy as np
import torch

SEED = 12

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)
torch.use_deterministic_algorithms(True)

In [4]:
# Rope

def compute_rope_params(seq_len, head_dim, device=None):
    # x: (seq_len, dim)

    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    theta = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))

    pos = torch.arange(seq_len).float()

    angles = pos[:, None] * theta[None, :]


    angles = angles[None, None, :, :]

    return torch.cos(angles), torch.sin(angles)



# similar to sebastian
def apply_rope(x, cos, sin, offset=0):

    batch_size, num_heads, seq_len, head_dim = x.shape   # (batch_size, num_heads, seq_len, head_dim)

    assert head_dim % 2 == 0, "Head dimension must be even"

    cos_sel = cos[...,offset : offset + seq_len, :].to(x.device, x.dtype)  #(1,1,seq_len,head_dim//2)
    sin_sel = sin[..., offset : offset + seq_len, :].to(x.device, x.dtype)

    x_even = x[..., 0::2]  # (b,n_heads,seq_len,head_dim//2)
    x_odd  = x[..., 1::2]


    x_rot = torch.empty_like(x)

    x_rot[..., 0::2] = x_even * cos_sel - x_odd * sin_sel
    x_rot[..., 1::2] = x_even * sin_sel + x_odd * cos_sel

    return x_rot.to(dtype=x.dtype)

In [5]:
import torch
from torch import nn


class causal_multi_head_transformer(nn.Module):
    def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads

        assert dout % num_heads == 0, f"d_out must be divisible by num_heads {dout // num_heads}"

        self.head_dim = dout // num_heads

        self.context_length = context_length



        self.wq = nn.Linear(din, dout, bias = qkv_bias)
        self.wk = nn.Linear(din, dout, bias = qkv_bias)
        self.wv = nn.Linear(din, dout, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(dout, dout)  # Linear layer to combine head outputs


        # Feedforward
        self.ff = nn.Sequential(nn.Linear(din, ff_dim),
                                            nn.ReLU(), nn.Linear(ff_dim, din),)
        self.norm1 = nn.RMSNorm(din)
        self.norm2 = nn.RMSNorm(din)

        self.apply_rope = apply_rope

        self.gate = nn.Linear(din, self.num_heads * self.head_dim)

        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, 2.0)  # sigmoid(2)  0.88
        # or for even closer to identity
        # nn.init.constant_(self.gate.bias, 4.0)  # sigmoid(4)  0.98

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

        self.qk_norm  = qk_norm
        self.pre_norm = pre_norm
        self.post_norm = post_norm


        if self.qk_norm:
            self.q_norm = nn.RMSNorm(self.head_dim)
            self.k_norm = nn.RMSNorm(self.head_dim)

        if self.pre_norm and self.post_norm:
          self.post_attn_norm = nn.RMSNorm(dout)
          self.post_ff_norm = nn.RMSNorm(dout)
          self.pre_ff_norm = nn.RMSNorm(dout)

    def forward(self, x, cos, sin, start_pos):

        b, num_tokens, d_in = x.shape

        assert num_tokens <= self.context_length

        #Pre-LN
        if self.pre_norm:
          x_norm = self.norm1(x)

        else:
          x_norm =x

        q = self.wq(x_norm)
        k = self.wk(x_norm)
        v = self.wv(x_norm)

        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)


        k = k.transpose(1,2)  # reshape to (b, num_heads, num_tokens, head_dim)
        v = v.transpose(1,2)
        q = q.transpose(1,2)

        #QK normalization (optional)
        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        q_enc = self.apply_rope(q, cos, sin, start_pos)
        k_enc = self.apply_rope(k, cos, sin, start_pos)

        attn_scores = q_enc @ k_enc.transpose(2,3)    # (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, num_tokens)

        attn_scores = attn_scores / (self.head_dim ** 0.5)

        mask = self.mask[:num_tokens, :num_tokens]
        masked = attn_scores.masked_fill(mask.bool(), float('-inf'))

        attn_weights = torch.softmax(masked, dim=-1)                             # shape: (b, num_heads, num_tokens, num_tokens)
        attn_weights = self.dropout(attn_weights)
        z = (attn_weights @ v)

        #gate applied elemnetwise

        gate_val = self.gate(x_norm)

        gate_val = gate_val.view(b, num_tokens, self.num_heads, self.head_dim)
        gate_val = gate_val.transpose(1,2)


        z = z * torch.sigmoid(gate_val)

       # Reshape and project
        z = z.transpose(1,2).contiguous().view(b, num_tokens, -1)  # Reshape to (b, num_tokens, num_heads * head_dim)
        attn_out = self.out_proj(z)

        # after attention output
        if self.pre_norm and self.post_norm:
          attn_out = self.post_attn_norm(attn_out)
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out

        elif self.pre_norm:                       #  Pre-LN
          x= x + attn_out
          x = x + self.ff(self.norm2(x))

        elif self.post_norm:                    #  Post-LN
          x = self.norm1(x + attn_out)
          x = self.norm2(x + self.ff(x))

        sink_val = attn_weights[:, :, :, 0].mean(dim=-1)

        return x, gate_val, sink_val

In [6]:
class Gated_Transformer_LM(nn.Module):
  def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, vocab_size, qk_norm, pre_norm, post_norm, n_transformer):
    super().__init__()


    assert din == dout

    self.embedding = nn.Embedding(vocab_size, din)


    self.dstack = nn.ModuleList([causal_multi_head_transformer(din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size)
    for _ in range(n_transformer)])

    cos, sin = compute_rope_params(context_length, din//num_heads)

    # register so they move with the model and are saved in state_dict
    self.register_buffer("rope_cos", cos)
    self.register_buffer("rope_sin", sin)

    self.cos, self.sin = cos, sin

    self.final_norm = nn.RMSNorm(din)
    self.out_head = nn.Linear(
        din, vocab_size, bias=False
    )


  def forward(self, inp, start_pos: int = 0):
      gate_vals = []
      attn_sink_val = []

      x = self.embedding(inp)

      for layer in self.dstack:
        x, gate_val, sink_info = layer(x, self.cos, self.sin, start_pos)

        gate_vals.append(gate_val)
        attn_sink_val.append(sink_info)

      x = self.final_norm(x)
      logits = self.out_head(x)


      return logits, gate_vals, attn_sink_val # Only return logits, as gate_val is not computed

### Model without Gating

In [7]:
import torch
from torch import nn

class causal_multi_head_transformer_no_gate(nn.Module):
    def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads

        assert dout % num_heads == 0, f"d_out must be divisible by num_heads {dout // num_heads}"

        self.head_dim = dout // num_heads

        self.context_length = context_length

        self.wq = nn.Linear(din, dout, bias = qkv_bias)
        self.wk = nn.Linear(din, dout, bias = qkv_bias)
        self.wv = nn.Linear(din, dout, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(dout, dout)  # Linear layer to combine head outputs


        # Feedforward
        self.ff = nn.Sequential( nn.Linear(din, ff_dim),
                                            nn.ReLU(), nn.Linear(ff_dim, din), )
        self.norm1 = nn.RMSNorm(din)
        self.norm2 = nn.RMSNorm(din)

        self.apply_rope = apply_rope

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

        self.qk_norm  = qk_norm
        self.pre_norm = pre_norm
        self.post_norm = post_norm


        if self.qk_norm:
          self.q_norm = nn.RMSNorm(self.head_dim)
          self.k_norm = nn.RMSNorm(self.head_dim)

        if self.pre_norm and self.post_norm:
          self.post_attn_norm = nn.RMSNorm(dout)
          self.post_ff_norm = nn.RMSNorm(dout)
          self.pre_ff_norm = nn.RMSNorm(dout)


    def forward(self, x, cos, sin, start_pos):

        b, num_tokens, d_in = x.shape

        assert num_tokens <= self.context_length

        #Pre-LN

        if self.pre_norm:
          x_norm = self.norm1(x)

        else:
          x_norm =x

        q = self.wq(x_norm)
        k = self.wk(x_norm)
        v = self.wv(x_norm)

        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)


        k = k.transpose(1,2)  # reshape to (b, num_heads, num_tokens, head_dim)
        v = v.transpose(1,2)
        q = q.transpose(1,2)

        #QK normalization (optional)
        # Qk normalization L2 norm return unit vectors q/|q| = 1, k/|k| = 1 which scales the values in q.k = cos (theta) and range of values [-1, 1]
        # and no magnitude scale is taken in consideration in SDPA q.k/root(head_dim) which limits the range of values to [-1/root(head_dim), 1/root(head_dim)].
        # can add learnable paramereter Y which can change the scale [-1/root(head_dim) * y, 1/root(head_dim)] * y]
        # if self.qk_norm:
        #     q = q / (q.norm(dim=-1, keepdim=True) + 1e-6)
        #     k = k / (k.norm(dim=-1, keepdim=True) + 1e-6)

        # QK norm with RMSNorm (common with LLM models)

        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)



        q_enc = self.apply_rope(q, cos, sin, start_pos)
        k_enc = self.apply_rope(k, cos, sin, start_pos)

        attn_scores = q_enc @ k_enc.transpose(2,3)
        attn_scores = attn_scores / (self.head_dim ** 0.5)

        mask = self.mask[:num_tokens, :num_tokens]
        masked = attn_scores.masked_fill(mask.bool(), float('-inf'))

        attn_weights = torch.softmax(masked, dim=-1)
        attn_weights = self.dropout(attn_weights)
        z = (attn_weights @ v)

        # Gating mechanism removed

       # Reshape and project
        z = z.transpose(1,2).contiguous().view(b, num_tokens, -1)  # Reshape to (b, num_tokens, num_heads, head_dim)
        attn_out = self.out_proj(z)

        # after attention output
        if self.pre_norm and self.post_norm:
          attn_out = self.post_attn_norm(attn_out)
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out


        elif self.pre_norm:                       #  Pre-LN
          x= x + attn_out
          x = x + self.ff(self.norm2(x))

        elif self.post_norm:                    #  Post-LN
          x = self.norm1(x + attn_out)
          x = self.norm2(x + self.ff(x))


        sink_val = attn_weights[:, :, :, 0].mean(dim=-1)

        return x,  sink_val # Only return x, as gate_val is not computed

In [8]:
class Transformer_No_Gate(nn.Module):
  def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, vocab_size, qk_norm, pre_norm, post_norm, n_transformer):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, din)


    self.dstack = nn.ModuleList([causal_multi_head_transformer_no_gate(din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, vocab_size)
    for _ in range(n_transformer)])

    cos, sin = compute_rope_params(context_length, din//num_heads)

    # register so they move with the model and are saved in state_dict
    self.register_buffer("rope_cos", cos)
    self.register_buffer("rope_sin", sin)

    self.cos, self.sin = cos, sin

    self.final_norm = nn.RMSNorm(din)
    self.out_head = nn.Linear(
        din, vocab_size, bias=False
    )


  def forward(self, inp, start_pos: int = 0):
      attn_sink_val = []
      x = self.embedding(inp)

      for layer in self.dstack:
        x, sink_info = layer(x, self.cos, self.sin, start_pos)
        attn_sink_val.append(sink_info)

      x = self.final_norm(x)
      logits = self.out_head(x)


      return logits, attn_sink_val # Only return logits, as gate_val is not computed


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from datasets import load_dataset
from transformers import AutoTokenizer

# -----------------------------
# Load Dataset
# -----------------------------
# Using the fully qualified Salesforce/wikitext path to avoid HfUriError
try:
    dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")
except Exception as e:
    print(f"Salesforce load failed: {e}. Attempting standard wikitext fallback...")
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1")

texts = dataset["train"]["text"]
texts = [t for t in texts if len(t.strip()) > 0]

# -----------------------------
# Tokenizer (FAST)
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# -----------------------------
# FAST Tokenization (BATCHED)
# -----------------------------
encodings = tokenizer(
    texts,
    padding=False,
    truncation=False
)

# Flatten tokens + add EOS between docs
all_tokens = []
for ids in encodings["input_ids"]:
    all_tokens.extend(ids + [tokenizer.eos_token_id])

tokens = torch.tensor(all_tokens, dtype=torch.long)

# -----------------------------
# Lazy Dataset (NO stacking)
# -----------------------------
class WikiTextDataset(Dataset):
    def __init__(self, tokens, seq_len=64, stride=None):
        self.tokens = tokens
        self.seq_len = seq_len
        self.stride = stride if stride is not None else seq_len
        self.num_samples = (len(tokens) - (seq_len + 1)) // self.stride

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        i = idx * self.stride
        chunk = self.tokens[i : i + self.seq_len + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

# -----------------------------
# Create Dataset
# -----------------------------
seq_len = 128
train_dataset = WikiTextDataset(
    tokens,
    seq_len=seq_len,
    stride=128
)

max_samples = 300000
train_dataset = Subset(train_dataset, range(min(max_samples, len(train_dataset))))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1063 > 1024). Running this sequence through the model will result in indexing errors


In [10]:
dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,      #  parallel loading
    pin_memory=True     #  faster GPU transfer
)


len(dataloader)

18750

In [11]:
vocab_size = tokenizer.vocab_size
print(vocab_size)

50257


In [12]:
outputs = tokenizer("a gaoal is good",  return_tensors="pt", add_special_tokens=True)['input_ids']
print(outputs)
print(tokenizer.decode(outputs, skip_special_tokens=False))

tensor([[  64,  308, 5488,  282,  318,  922]])
['a gaoal is good']


In [13]:
import math
import copy
import torch
from torch.utils.data import DataLoader, random_split

device = "cuda" if torch.cuda.is_available() else "cpu"

model_no_gate = Transformer_No_Gate(din=256, dout=256, context_length=128,  dropout=0.1, ff_dim=1152, num_heads=8,
                                    vocab_size = 50257, qk_norm = False, pre_norm = True, post_norm = False, n_transformer = 8).to(device)

model_with_gate = Gated_Transformer_LM(din=256, dout=256, context_length=128, dropout=0.1, ff_dim=1024, num_heads=8,
                             vocab_size=50257, qk_norm = False, pre_norm = True, post_norm = False, n_transformer = 8).to(device)

In [14]:
!pip install torchinfo

In [15]:
from torchinfo import summary

# Providing a dummy input with the correct shape (batch_size, sequence_length)
# Based on your initialization, sequence_length is 128
batch_size = 16
sequence_length = 128
dummy_input = torch.zeros((batch_size, sequence_length), dtype=torch.long).to(device)

print("Summary for Transformer_No_Gate:")
summary(model_no_gate, input_data=dummy_input, depth=4)

Summary for Transformer_No_Gate:


Layer (type:depth-idx)                                  Output Shape              Param #
Transformer_No_Gate                                     [16, 128, 50257]          --
├─Embedding: 1-1                                        [16, 128, 256]            12,865,792
├─ModuleList: 1-2                                       --                        --
│    └─causal_multi_head_transformer_no_gate: 2-1       [16, 128, 256]            --
│    │    └─RMSNorm: 3-1                                [16, 128, 256]            256
│    │    └─Linear: 3-2                                 [16, 128, 256]            65,536
│    │    └─Linear: 3-3                                 [16, 128, 256]            65,536
│    │    └─Linear: 3-4                                 [16, 128, 256]            65,536
│    │    └─Dropout: 3-5                                [16, 8, 128, 128]         --
│    │    └─Linear: 3-6                                 [16, 128, 256]            65,792
│    │    └─RMSNorm: 3-7           

In [16]:
summary(model_with_gate, input_data=dummy_input, depth=4)

Layer (type:depth-idx)                        Output Shape              Param #
Gated_Transformer_LM                          [16, 128, 50257]          --
├─Embedding: 1-1                              [16, 128, 256]            12,865,792
├─ModuleList: 1-2                             --                        --
│    └─causal_multi_head_transformer: 2-1     [16, 128, 256]            --
│    │    └─RMSNorm: 3-1                      [16, 128, 256]            256
│    │    └─Linear: 3-2                       [16, 128, 256]            65,536
│    │    └─Linear: 3-3                       [16, 128, 256]            65,536
│    │    └─Linear: 3-4                       [16, 128, 256]            65,536
│    │    └─Dropout: 3-5                      [16, 8, 128, 128]         --
│    │    └─Linear: 3-6                       [16, 128, 256]            65,792
│    │    └─Linear: 3-7                       [16, 128, 256]            65,792
│    │    └─RMSNorm: 3-8                      [16, 128, 256]      

### Training without Gating for Param Match baseline Comparison with Gated Model

In [17]:
import math
import copy
from torch.utils.data import DataLoader, random_split

def model_training_no_gate(dataloader_dataset, qk_norm, pre_norm, post_norm, vocab_size, n_transformer, seed,
                          val_ratio=0.2):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(device)

    set_seed(seed)

    gen = torch.Generator()
    gen.manual_seed(seed)

    val_size = int(len(dataloader_dataset) * val_ratio)
    train_size = len(dataloader_dataset) - val_size
    split_gen = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
      dataloader_dataset, [train_size, val_size], generator=split_gen
    )

    train_dataloader = DataLoader(
      train_dataset, batch_size=16, shuffle=True,
      num_workers=0,
      pin_memory=True,
      generator=gen, worker_init_fn=seed_worker
    )

    print('train_dataloader size: ', len(train_dataloader))

    val_dataloader = DataLoader(
      val_dataset, batch_size=16, shuffle=False,
      num_workers=0, pin_memory=True
    )

    print('val_dataloader size: ', len(val_dataloader))
    # ----------------------------------------------------------------------

    ## using ff_dim =1152 for param match baseline comparison with gated model
    model_no_gate = Transformer_No_Gate(din=256, dout=256, context_length=128,  dropout=0.1, ff_dim=1152, num_heads=8,
                                        vocab_size = vocab_size, qk_norm = qk_norm, pre_norm = pre_norm, post_norm = post_norm, n_transformer = n_transformer).to(device)

    ## ff_dim = 1024 (for all the runs in experiments conducted)
    model_gate = Gated_Transformer_LM(din=256, dout=256, context_length=128,  dropout=0.1, ff_dim=1024, num_heads=8,
                                    vocab_size = vocab_size, qk_norm = qk_norm, pre_norm = pre_norm, post_norm = post_norm, n_transformer = n_transformer).to(device)

    print(f'Total Model parameters for no gate model : {sum(p.numel() for p in model_no_gate.parameters())}')
    print(f'Total Model parameters for gated model : {sum(p.numel() for p in model_gate.parameters())}')

    optimizer_no_gate = torch.optim.Adam(model_no_gate.parameters(), lr=1e-3)
    loss_fn_no_gate = nn.CrossEntropyLoss()

    loss_history_no_gate, max_act_history_no_gate, grad_norm_history_no_gate = [], [], []

    epoch_loss_no_gate = []
    val_epoch_loss_no_gate = []

    train_ppl_history_no_gate = []
    val_ppl_history_no_gate = []
    best_val_ppl_no_gate = float("inf")
    best_model_state_no_gate = copy.deepcopy(model_no_gate.state_dict())
    epochs_without_improvement_no_gate = 0
    patience_no_gate = 1

    print("\n--- Training without Gating ---")
    for epoch in range(10):
        model_no_gate.train()
        total_loss_no_gate=0

        for step, (xb, yb) in enumerate(train_dataloader):
            xb, yb = xb.to(device), yb.to(device)
            optimizer_no_gate.zero_grad()
            out_no_gate, attn_sink_info = model_no_gate(xb)

            # Reshape for CrossEntropyLoss: (batch * seq, vocab)
            loss_no_gate = loss_fn_no_gate(out_no_gate.reshape(-1, vocab_size), yb.reshape(-1))
            loss_no_gate.backward()

            total_norm_no_gate = torch.nn.utils.clip_grad_norm_(model_no_gate.parameters(), max_norm=1.0)
            optimizer_no_gate.step()

            loss_history_no_gate.append(loss_no_gate.item())
            max_act_history_no_gate.append(out_no_gate.abs().max().item())
            grad_norm_history_no_gate.append(total_norm_no_gate.item())

            total_loss_no_gate += loss_no_gate.item()

            if step % 500 == 0:
              out_no_gate_mean = out_no_gate.mean().item()
              max_act = out_no_gate.abs().max().item()

              # Accessing sink info for the last layer: attn_sink_info[-1]
              # attn_sink_info is a list of tensors
              sink_val_tensor = attn_sink_info[-1]
              avg_sink_val = sink_val_tensor.float().mean().item()

              print(f"Epoch {epoch} Step {step} | Loss={loss_no_gate.item():.4f} | "
                    f"MaxAct={max_act:.4f} | MeanAct={out_no_gate_mean:.4f} | "
                    f"GradNorm={total_norm_no_gate:.4f} | Sink Val={avg_sink_val:.4f}")



        avg_train_loss_no_gate = total_loss_no_gate / len(train_dataloader)
        epoch_loss_no_gate.append(avg_train_loss_no_gate)

        train_ppl_no_gate = math.exp(min(avg_train_loss_no_gate, 20))
        train_ppl_history_no_gate.append(train_ppl_no_gate)

        # ===================== Validation =====================
        model_no_gate.eval()
        total_val_loss_no_gate = 0.0

        with torch.no_grad():
            for step, (xb, yb) in enumerate(val_dataloader):
                xb, yb = xb.to(device), yb.to(device)

                val_out_no_gate, attn_sink_info = model_no_gate(xb)
                val_loss_no_gate = loss_fn_no_gate(val_out_no_gate.reshape(-1, vocab_size), yb.reshape(-1))
                total_val_loss_no_gate += val_loss_no_gate.item()

        avg_val_loss_no_gate = total_val_loss_no_gate / len(val_dataloader)
        val_epoch_loss_no_gate.append(avg_val_loss_no_gate)

        val_ppl_no_gate = math.exp(min(avg_val_loss_no_gate, 20))
        val_ppl_history_no_gate.append(val_ppl_no_gate)

        print(
            f"\nEpoch {epoch} Summary | "
            f"Train Loss={avg_train_loss_no_gate:.4f} | Train PPL={train_ppl_no_gate:.2f} | "
            f"Val Loss={avg_val_loss_no_gate:.4f} | Val PPL={val_ppl_no_gate:.2f}\n"
        )

        if val_ppl_no_gate < best_val_ppl_no_gate:
            best_val_ppl_no_gate = val_ppl_no_gate
            best_model_state_no_gate = copy.deepcopy(model_no_gate.state_dict())
            epochs_without_improvement_no_gate = 0
        else:
            epochs_without_improvement_no_gate += 1
            if epochs_without_improvement_no_gate >= patience_no_gate:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    # Restore Best Model
    model_no_gate.load_state_dict(best_model_state_no_gate)

    # Updated return dictionary with _no_gate suffix
    return {
        "model_no_gate": model_no_gate,
        "train_loss_no_gate": epoch_loss_no_gate,
        "val_loss_no_gate": val_epoch_loss_no_gate,
        "train_ppl_no_gate": train_ppl_history_no_gate,
        "val_ppl_no_gate": val_ppl_history_no_gate,
    }

### config 1: pre norm =True, post_norm = False, qk norm = False

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32564992
Total Model parameters for gated model : 32566016

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9946 | MaxAct=3.3038 | MeanAct=-0.0003 | GradNorm=1.7519 | Sink Val=0.0423
Epoch 0 Step 500 | Loss=5.9758 | MaxAct=15.6722 | MeanAct=-3.7753 | GradNorm=0.5793 | Sink Val=0.0260
Epoch 0 Step 1000 | Loss=5.6138 | MaxAct=17.1031 | MeanAct=-3.7981 | GradNorm=0.5423 | Sink Val=0.0251
Epoch 0 Step 1500 | Loss=5.3483 | MaxAct=18.0088 | MeanAct=-3.9738 | GradNorm=0.5833 | Sink Val=0.0250
Epoch 0 Step 2000 | Loss=5.5426 | MaxAct=18.4064 | MeanAct=-3.8420 | GradNorm=0.5997 | Sink Val=0.0268
Epoch 0 Step 2500 | Loss=5.0661 | MaxAct=19.1668 | MeanAct=-3.9241 | GradNorm=0.5672 | Sink Val=0.0240
Epoch 0 Step 3000 | Loss=4.9513 | MaxAct=20.2634 | MeanAct=-3.9911 | GradNorm=0.5702 | Sink Val=0.0227
Epoch 0 Step 3500 | Loss=4.7727 | MaxAct=19.4752 | MeanAct=-4.0081 | GradNorm=0.5535 |

In [ ]:
print('#### Logs for Config 1 (No Gate) - SEED 12')
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

#### Logs for Config 1 (No Gate) - SEED 12
train_loss_no_gate:  [4.721040027268727, 4.072449858268102, 3.877614791202545, 3.764118768723806, 3.6857507258097333, 3.6265562361558277, 3.5798137228329976, 3.54106564745903, 3.5080602185249328, 3.47975887521108]
val_loss_no_gate:  [4.238985468419393, 4.011365624554952, 3.910367743428548, 3.8507885897954304, 3.815004820124308, 3.7886522839864094, 3.769806240208944, 3.755475228436788, 3.745386950556437, 3.7362797693252565]
train_ppl_no_gate:  [112.28497139407746, 58.70059471819146, 48.30885084612399, 43.12568540712636, 39.87504644262794, 37.58316595411926, 35.86685904785606, 34.50366835306167, 33.38344834182601, 32.45189617577702]
val_ppl_no_gate:  [69.3374710889061, 55.22223192998839, 49.91730535954925, 47.030136097420346, 45.37697510508085, 44.196795390878506, 43.371660337806205, 42.754533153387094, 42.32538188061136, 41.941666890255284]

Best Training PPL:  32.45189617577702
Best Validation PPL:  41.941666890255284


In [ ]:
SEED2 = 42
model_det_no_gate2 = model_training_no_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED2)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32564992
Total Model parameters for gated model : 32566016

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9524 | MaxAct=3.3978 | MeanAct=0.0020 | GradNorm=1.6628 | Sink Val=0.0423
Epoch 0 Step 500 | Loss=5.8287 | MaxAct=16.0257 | MeanAct=-3.8735 | GradNorm=0.5259 | Sink Val=0.0326
Epoch 0 Step 1000 | Loss=5.7488 | MaxAct=17.4525 | MeanAct=-3.7000 | GradNorm=0.5940 | Sink Val=0.0337
Epoch 0 Step 1500 | Loss=5.3985 | MaxAct=17.7610 | MeanAct=-3.9747 | GradNorm=0.5524 | Sink Val=0.0320
Epoch 0 Step 2000 | Loss=5.0757 | MaxAct=18.8049 | MeanAct=-3.8706 | GradNorm=0.5268 | Sink Val=0.0285
Epoch 0 Step 2500 | Loss=5.0087 | MaxAct=19.5239 | MeanAct=-3.8688 | GradNorm=0.5597 | Sink Val=0.0311
Epoch 0 Step 3000 | Loss=4.7893 | MaxAct=20.9519 | MeanAct=-3.8720 | GradNorm=0.5347 | Sink Val=0.0323
Epoch 0 Step 3500 | Loss=4.8607 | MaxAct=20.7907 | MeanAct=-3.8172 | GradNorm=0.5659 | 

In [ ]:
print('#### Logs for Config 1 (No Gate) - SEED 42')
print('train_loss_no_gate: ', model_det_no_gate2['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate2['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate2['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate2['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate2['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate2['val_ppl_no_gate']))

#### Logs for Config 1 (No Gate) - SEED 42
train_loss_no_gate:  [4.721580334409078, 4.071011502869924, 3.871379057041804, 3.7562223584810894, 3.6768893197695416, 3.617390494569143, 3.56982127327919, 3.5307553338527677, 3.4976755469640097, 3.4688170148213704]
val_loss_no_gate:  [4.23294583791097, 4.005325730832418, 3.9034625504811604, 3.845393350156148, 3.8072707269668578, 3.7816091124216715, 3.762240054130554, 3.7492038685480753, 3.737138701756795, 3.7293668190002442]
train_ppl_no_gate:  [112.3456561586052, 58.61622309363925, 48.008546975283195, 42.786488285469254, 39.52325843540993, 37.2402622509721, 35.51024595462389, 34.14975233949637, 33.0385660393215, 32.09874763283226]
val_ppl_no_gate:  [68.91996045724687, 54.889700755390024, 49.573804070858344, 46.77708050453065, 45.027378999463075, 43.88660342822212, 43.044740612499815, 42.487243101205834, 41.97770742415604, 41.652726097846006]

Best Training PPL:  32.09874763283226
Best Validation PPL:  41.652726097846006


In [ ]:
SEED3 = 100
model_det_no_gate3 = model_training_no_gate(train_dataset, qk_norm = False, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED3)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32564992
Total Model parameters for gated model : 32566016

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0025 | MaxAct=3.3248 | MeanAct=0.0011 | GradNorm=1.5736 | Sink Val=0.0433
Epoch 0 Step 500 | Loss=5.9880 | MaxAct=16.0121 | MeanAct=-3.6082 | GradNorm=0.5451 | Sink Val=0.0251
Epoch 0 Step 1000 | Loss=5.5575 | MaxAct=17.1129 | MeanAct=-3.8146 | GradNorm=0.5682 | Sink Val=0.0228
Epoch 0 Step 1500 | Loss=5.2793 | MaxAct=18.8273 | MeanAct=-3.7291 | GradNorm=0.5864 | Sink Val=0.0212
Epoch 0 Step 2000 | Loss=5.3140 | MaxAct=18.9720 | MeanAct=-3.7411 | GradNorm=0.5710 | Sink Val=0.0269
Epoch 0 Step 2500 | Loss=5.0980 | MaxAct=18.8401 | MeanAct=-3.8806 | GradNorm=0.5643 | Sink Val=0.0264
Epoch 0 Step 3000 | Loss=4.9901 | MaxAct=19.4402 | MeanAct=-3.9993 | GradNorm=0.5809 | Sink Val=0.0215
Epoch 0 Step 3500 | Loss=4.9258 | MaxAct=20.8818 | MeanAct=-3.9803 | GradNorm=0.5754 | 

In [ ]:
print('#### Logs for Config 1 (No Gate) - SEED 100')
print('train_loss_no_gate: ', model_det_no_gate3['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate3['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate3['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate3['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate3['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate3['val_ppl_no_gate']))

#### Logs for Config 1 (No Gate) - SEED 100
train_loss_no_gate:  [4.714074929618835, 4.064568695926666, 3.869021163606644, 3.7557716554482776, 3.6773719920635224, 3.618031811841329, 3.5711632491906484, 3.532437051375707, 3.4992499546527864, 3.470830533806483]
val_loss_no_gate:  [4.230977048619589, 4.007818765068055, 3.9082319971720376, 3.855540322113037, 3.813430987993876, 3.7847015657424925, 3.768950889968872, 3.7572965673446657, 3.7445895992279055, 3.7345917086283364]
train_ppl_no_gate:  [111.50561290840292, 58.23978404870721, 47.895481288343, 42.767208630461475, 39.54233982188238, 37.26415273424536, 35.557931838802446, 34.20723089420945, 33.090623180548576, 32.16344418259686]
val_ppl_no_gate:  [68.78440506092842, 55.02671337617061, 49.81080842718618, 47.254142502767905, 45.30561553240229, 44.02253076688241, 43.33457823907583, 42.83247460546101, 42.291647132148796, 41.87092653461972]

Best Training PPL:  32.16344418259686
Best Validation PPL:  41.87092653461972


### config 2: pre norm = True, post_norm = False, qk norm = True

In [ ]:
print(SEED)

model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

12
cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32565504
Total Model parameters for gated model : 32566528

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9983 | MaxAct=3.3587 | MeanAct=-0.0003 | GradNorm=1.7372 | Sink Val=0.0418
Epoch 0 Step 500 | Loss=5.9440 | MaxAct=15.6935 | MeanAct=-3.7035 | GradNorm=0.5583 | Sink Val=0.0308
Epoch 0 Step 1000 | Loss=5.6064 | MaxAct=17.1115 | MeanAct=-3.8114 | GradNorm=0.5513 | Sink Val=0.0302
Epoch 0 Step 1500 | Loss=5.3345 | MaxAct=18.1424 | MeanAct=-3.9676 | GradNorm=0.5895 | Sink Val=0.0288
Epoch 0 Step 2000 | Loss=5.5231 | MaxAct=17.8941 | MeanAct=-3.8351 | GradNorm=0.6424 | Sink Val=0.0278
Epoch 0 Step 2500 | Loss=5.0586 | MaxAct=18.8646 | MeanAct=-3.9839 | GradNorm=0.6086 | Sink Val=0.0304
Epoch 0 Step 3000 | Loss=4.9528 | MaxAct=20.4920 | MeanAct=-4.0902 | GradNorm=0.5682 | Sink Val=0.0254
Epoch 0 Step 3500 | Loss=4.7514 | MaxAct=20.2452 | MeanAct=-4.0446 | GradNorm=0.556

In [ ]:
print('#### Logs for Config 2 (No Gate) - SEED 12')
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

#### Logs for Config 2 (No Gate) - SEED 12
train_loss_no_gate:  [4.706265795389811, 4.055015884256363, 3.8593051284948983, 3.7460330294768016, 3.6679407786210376, 3.6088894996325176, 3.5621547252496084, 3.523230013656616, 3.490466825087865, 3.4618341325918833]
val_loss_no_gate:  [4.222488624191284, 3.9953777857462565, 3.894386366144816, 3.8371504368464153, 3.8000631392796835, 3.772644000244141, 3.7555014695485434, 3.744039668718974, 3.732585442860921, 3.7219055228551228]
train_ppl_no_gate:  [110.63824172677647, 57.686079289357764, 47.43238050529112, 42.35273625323105, 39.17116066124463, 36.92502478462493, 35.239045861077734, 33.893729055342725, 32.80125658215625, 31.87538661783088]
val_ppl_no_gate:  [68.20300491331211, 54.34636802929041, 49.12589879703072, 46.39308587649153, 44.70400698299478, 43.49491350091129, 42.75565509459006, 42.26839605894109, 41.78700653911931, 41.343099318160355]

Best Training PPL:  31.87538661783088
Best Validation PPL:  41.343099318160355


In [ ]:
SEED2 = 42
model_det_no_gate2 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED2)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32565504
Total Model parameters for gated model : 32566528

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9532 | MaxAct=3.4730 | MeanAct=0.0019 | GradNorm=1.6507 | Sink Val=0.0422
Epoch 0 Step 500 | Loss=5.8078 | MaxAct=15.9418 | MeanAct=-3.8995 | GradNorm=0.5582 | Sink Val=0.0302
Epoch 0 Step 1000 | Loss=5.7150 | MaxAct=17.1683 | MeanAct=-3.7633 | GradNorm=0.5844 | Sink Val=0.0335
Epoch 0 Step 1500 | Loss=5.3940 | MaxAct=17.9863 | MeanAct=-3.9678 | GradNorm=0.5653 | Sink Val=0.0313
Epoch 0 Step 2000 | Loss=5.0673 | MaxAct=18.4380 | MeanAct=-3.8537 | GradNorm=0.5317 | Sink Val=0.0308
Epoch 0 Step 2500 | Loss=4.9906 | MaxAct=19.1992 | MeanAct=-3.8819 | GradNorm=0.5592 | Sink Val=0.0305
Epoch 0 Step 3000 | Loss=4.7803 | MaxAct=20.1370 | MeanAct=-3.9163 | GradNorm=0.5393 | Sink Val=0.0298
Epoch 0 Step 3500 | Loss=4.8483 | MaxAct=20.7722 | MeanAct=-3.8930 | GradNorm=0.5626 | 

In [ ]:
print('#### Logs for Config 2 (No Gate) - SEED 42')
print('train_loss_no_gate: ', model_det_no_gate2['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate2['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate2['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate2['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate2['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate2['val_ppl_no_gate']))

#### Logs for Config 2 (No Gate) - SEED 42
train_loss_no_gate:  [4.706393399540583, 4.052265752601624, 3.8561204610506694, 3.7426563688437144, 3.664320768292745, 3.6053987387657167, 3.558252109384537, 3.5195357182343803, 3.4864583292643228, 3.457876995611191]
val_loss_no_gate:  [4.213860936228434, 3.987481098620097, 3.8859595251083374, 3.83102642771403, 3.795168645095825, 3.768358670552572, 3.7499446547190347, 3.7388419537862143, 3.72295132522583, 3.7183566018422445]
train_ppl_no_gate:  [110.6523605264445, 57.52765292319972, 47.2815644242913, 42.209966613984456, 39.02961700440079, 36.796353064862906, 35.101789405446944, 33.7687466104329, 32.67003605630931, 31.74950058507941]
val_ppl_no_gate:  [67.61710179782112, 53.918901769489565, 48.71366202343913, 46.10984237302305, 44.485738074511545, 43.308922258180324, 42.5187287239545, 42.04926696418256, 41.38635864585066, 41.196635971400596]

Best Training PPL:  31.74950058507941
Best Validation PPL:  41.196635971400596


In [ ]:
SEED3 = 100
model_det_no_gate3 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED3)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32565504
Total Model parameters for gated model : 32566528

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0025 | MaxAct=3.2502 | MeanAct=0.0011 | GradNorm=1.5603 | Sink Val=0.0435
Epoch 0 Step 500 | Loss=5.9919 | MaxAct=16.0730 | MeanAct=-3.6322 | GradNorm=0.5571 | Sink Val=0.0263
Epoch 0 Step 1000 | Loss=5.5188 | MaxAct=17.0862 | MeanAct=-3.7862 | GradNorm=0.5474 | Sink Val=0.0281
Epoch 0 Step 1500 | Loss=5.2636 | MaxAct=18.1516 | MeanAct=-3.7461 | GradNorm=0.5738 | Sink Val=0.0233
Epoch 0 Step 2000 | Loss=5.3065 | MaxAct=18.4305 | MeanAct=-3.7626 | GradNorm=0.5710 | Sink Val=0.0282
Epoch 0 Step 2500 | Loss=5.0876 | MaxAct=19.2473 | MeanAct=-3.9075 | GradNorm=0.5450 | Sink Val=0.0276
Epoch 0 Step 3000 | Loss=4.9744 | MaxAct=20.2680 | MeanAct=-4.0012 | GradNorm=0.5838 | Sink Val=0.0246
Epoch 0 Step 3500 | Loss=4.9042 | MaxAct=20.5196 | MeanAct=-3.9834 | GradNorm=0.5440 | 

In [ ]:
print('#### Logs for Config 2 (No Gate) - SEED 100')
print('train_loss_no_gate: ', model_det_no_gate3['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate3['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate3['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate3['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate3['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate3['val_ppl_no_gate']))

#### Logs for Config 2 (No Gate) - SEED 100
train_loss_no_gate:  [4.703060619481405, 4.049422975508372, 3.8544833018779756, 3.741511072063446, 3.663535722796122, 3.60440544907252, 3.557353473583857, 3.5185238812287647, 3.485540781434377, 3.456833696715037]
val_loss_no_gate:  [4.215140083948771, 3.9935479848861695, 3.893815070915222, 3.8390060437520344, 3.8002756072362263, 3.774171057764689, 3.7566023367563885, 3.7441920576731365, 3.73036910572052, 3.724017726580302]
train_ppl_no_gate:  [110.28419439493307, 57.364346860586274, 47.2042203069857, 42.16165134806674, 38.99898900309681, 36.7598217726979, 35.07025984973485, 33.734595423610216, 32.64007348377041, 31.716393639388496]
val_ppl_no_gate:  [67.70364940122553, 54.24701592333293, 49.097841420684205, 46.479253128566114, 44.713506161107475, 43.56138347449817, 42.80274931071658, 42.27483778642223, 41.69449500078885, 41.430516655663254]

Best Training PPL:  31.716393639388496
Best Validation PPL:  41.430516655663254


### config 3: pre_norm = True, post norm = True, qk norm = True

In [ ]:
model_det_no_gate = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32571648
Total Model parameters for gated model : 32572672

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0089 | MaxAct=3.1659 | MeanAct=-0.0007 | GradNorm=3.4371 | Sink Val=0.0463
Epoch 0 Step 500 | Loss=6.0053 | MaxAct=15.6627 | MeanAct=-3.7224 | GradNorm=0.6098 | Sink Val=0.0326
Epoch 0 Step 1000 | Loss=5.6376 | MaxAct=17.5322 | MeanAct=-3.8000 | GradNorm=0.5767 | Sink Val=0.0318
Epoch 0 Step 1500 | Loss=5.3314 | MaxAct=17.9732 | MeanAct=-3.9091 | GradNorm=0.6280 | Sink Val=0.0318
Epoch 0 Step 2000 | Loss=5.5226 | MaxAct=18.6166 | MeanAct=-3.7985 | GradNorm=0.6204 | Sink Val=0.0294
Epoch 0 Step 2500 | Loss=5.0734 | MaxAct=19.8933 | MeanAct=-3.8947 | GradNorm=0.6165 | Sink Val=0.0292
Epoch 0 Step 3000 | Loss=4.9621 | MaxAct=19.8525 | MeanAct=-4.0101 | GradNorm=0.6313 | Sink Val=0.0274
Epoch 0 Step 3500 | Loss=4.7374 | MaxAct=20.0686 | MeanAct=-3.9462 | GradNorm=0.5753 |

In [ ]:
print('#### Logs for Config 3 (No Gate) - SEED 12')
print('train_loss_no_gate: ', model_det_no_gate['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate['val_ppl_no_gate']))

#### Logs for Config 3 (No Gate) - SEED 12
train_loss_no_gate:  [4.720964284324646, 4.06525154967308, 3.861221272468567, 3.7414743659973144, 3.6579397126197817, 3.5941715816815694, 3.5430923386891684, 3.500162666098277, 3.463308565616608, 3.431123368024826]
val_loss_no_gate:  [4.241296324284871, 4.004560901069641, 3.9001322735468547, 3.837901377105713, 3.7983409799575805, 3.768659628868103, 3.7522604055404662, 3.738644987487793, 3.726145285161336, 3.713844663302104]
train_ppl_no_gate:  [112.27646692184894, 58.279566884818706, 47.52335490754421, 42.16010378810674, 38.78135975942695, 36.38554504710783, 34.57366754453415, 33.120839158200596, 31.922419405363073, 30.91134807263134]
val_ppl_no_gate:  [69.49788526630569, 54.84773552875523, 49.408984174895544, 46.42793739649621, 44.62708581489828, 43.32195840003986, 42.61730560061158, 42.0409854913296, 41.51875634115593, 41.01117798398062]

Best Training PPL:  30.91134807263134
Best Validation PPL:  41.01117798398062


In [ ]:
SEED2 = 42
model_det_no_gate2 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED2)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32571648
Total Model parameters for gated model : 32572672

--- Training without Gating ---
Epoch 0 Step 0 | Loss=10.9355 | MaxAct=3.1489 | MeanAct=0.0038 | GradNorm=3.5891 | Sink Val=0.0434
Epoch 0 Step 500 | Loss=5.8477 | MaxAct=15.9343 | MeanAct=-3.8833 | GradNorm=0.5774 | Sink Val=0.0329
Epoch 0 Step 1000 | Loss=5.7503 | MaxAct=16.9881 | MeanAct=-3.6070 | GradNorm=0.6634 | Sink Val=0.0350
Epoch 0 Step 1500 | Loss=5.3980 | MaxAct=18.1565 | MeanAct=-3.9043 | GradNorm=0.6093 | Sink Val=0.0329
Epoch 0 Step 2000 | Loss=5.0742 | MaxAct=18.9282 | MeanAct=-3.8073 | GradNorm=0.5377 | Sink Val=0.0271
Epoch 0 Step 2500 | Loss=4.9794 | MaxAct=19.6994 | MeanAct=-3.8218 | GradNorm=0.5742 | Sink Val=0.0287
Epoch 0 Step 3000 | Loss=4.7761 | MaxAct=20.1542 | MeanAct=-3.8237 | GradNorm=0.5468 | Sink Val=0.0270
Epoch 0 Step 3500 | Loss=4.8714 | MaxAct=20.8732 | MeanAct=-3.8290 | GradNorm=0.5744 | 

In [ ]:
print('#### Logs for Config 3 (No Gate) - SEED 42')
print('train_loss_no_gate: ', model_det_no_gate2['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate2['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate2['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate2['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate2['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate2['val_ppl_no_gate']))

#### Logs for Config 3 (No Gate) - SEED 42
train_loss_no_gate:  [4.720328264458974, 4.060749308029811, 3.856300242757797, 3.736923016866048, 3.653599439462026, 3.590217742697398, 3.5389202399253845, 3.4960622326215107, 3.459331674353282, 3.4271902399698893]
val_loss_no_gate:  [4.231297085889181, 3.9957023089726764, 3.893268139139811, 3.8316328932444255, 3.794756972885132, 3.7658827943166098, 3.7444608825683594, 3.7326292825698855, 3.7206976617177325, 3.7119933654149375]
train_ppl_no_gate:  [112.20507956273495, 58.01776797504082, 47.29006554880961, 41.96865444324714, 38.61340281780845, 36.24196649093163, 34.42972327256769, 32.98530741987369, 31.79571951735137, 30.790008560652048]
val_ppl_no_gate:  [68.80642215707239, 54.36400755004977, 49.0709955931944, 46.13781488435886, 44.46742830144506, 43.201827358322255, 42.28620384293176, 41.78883850948071, 41.293192741369495, 40.93532431236414]

Best Training PPL:  30.790008560652048
Best Validation PPL:  40.93532431236414


In [18]:
SEED3 = 100
model_det_no_gate3 = model_training_no_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, vocab_size = vocab_size,
                                                                              n_transformer = 8, seed=SEED3)

cuda
train_dataloader size:  15000
val_dataloader size:  3750
Total Model parameters for no gate model : 32571648
Total Model parameters for gated model : 32572672

--- Training without Gating ---
Epoch 0 Step 0 | Loss=11.0663 | MaxAct=3.2896 | MeanAct=0.0021 | GradNorm=3.0934 | Sink Val=0.0469
Epoch 0 Step 500 | Loss=6.0434 | MaxAct=16.0286 | MeanAct=-3.5038 | GradNorm=0.6066 | Sink Val=0.0296
Epoch 0 Step 1000 | Loss=5.5813 | MaxAct=16.8565 | MeanAct=-3.6846 | GradNorm=0.5955 | Sink Val=0.0311
Epoch 0 Step 1500 | Loss=5.2671 | MaxAct=17.9751 | MeanAct=-3.6790 | GradNorm=0.6166 | Sink Val=0.0297
Epoch 0 Step 2000 | Loss=5.3342 | MaxAct=18.6325 | MeanAct=-3.7368 | GradNorm=0.6328 | Sink Val=0.0293
Epoch 0 Step 2500 | Loss=5.0951 | MaxAct=20.0772 | MeanAct=-3.8063 | GradNorm=0.5701 | Sink Val=0.0273
Epoch 0 Step 3000 | Loss=4.9686 | MaxAct=20.7887 | MeanAct=-3.8542 | GradNorm=0.5814 | Sink Val=0.0257
Epoch 0 Step 3500 | Loss=4.8930 | MaxAct=20.6427 | MeanAct=-3.8945 | GradNorm=0.5450 | 

In [19]:
print('#### Logs for Config 3 (No Gate) - SEED 100')
print('train_loss_no_gate: ', model_det_no_gate3['train_loss_no_gate'])
print('val_loss_no_gate: ', model_det_no_gate3['val_loss_no_gate'])
print('train_ppl_no_gate: ', model_det_no_gate3['train_ppl_no_gate'])
print('val_ppl_no_gate: ', model_det_no_gate3['val_ppl_no_gate'])
print('\nBest Training PPL: ', min(model_det_no_gate3['train_ppl_no_gate']))
print('Best Validation PPL: ', min(model_det_no_gate3['val_ppl_no_gate']))

#### Logs for Config 3 (No Gate) - SEED 100
train_loss_no_gate:  [4.713269011465709, 4.056574373133977, 3.8549560453573863, 3.7369938990910847, 3.6542100248972575, 3.5911188117822013, 3.540206759119034, 3.4976406685988106, 3.461083945051829, 3.429145662021637]
val_loss_no_gate:  [4.229502647654216, 4.003851519457499, 3.8994026774724326, 3.841207585779826, 3.8005665196100873, 3.770984869194031, 3.7534657709121704, 3.7416758008321125, 3.7305772085825604, 3.718945100212097]
train_ppl_no_gate:  [111.4157847127333, 57.77605249523248, 47.22654106991836, 41.9716293802897, 38.636986798458565, 36.274637723817506, 34.474046277474784, 33.037413728245006, 31.851483067149754, 30.850274926251938]
val_ppl_no_gate:  [68.68306399474532, 54.808841350741915, 49.37294872126162, 46.58169187798016, 44.72651576556416, 43.42280957087479, 42.668705996923464, 42.16859715704983, 41.703172647420075, 41.220887259735086]

Best Training PPL:  30.850274926251938
Best Validation PPL:  41.220887259735086


In [ ]:
### Summary of Best PPLs (No Gate Models)

# Config 1 (pre_norm=True, post_norm=False, qk_norm=False)
# SEED 12 : Best Train PPL = 32.45 | Best Val PPL = 41.94
# SEED 42 : Best Train PPL = 32.10 | Best Val PPL = 41.65
# SEED 100 : Best Train PPL = 32.16 | Best Val PPL = 41.87
# => Average : Train PPL = 32.24 ± 0.19 | Val PPL = 41.82 ± 0.15

# Config 2 (pre_norm=True, post_norm=False, qk_norm=True)
# SEED 12 : Best Train PPL = 31.88 | Best Val PPL = 41.34
# SEED 42 : Best Train PPL = 31.75 | Best Val PPL = 41.20
# SEED 100 : Best Train PPL = 31.72 | Best Val PPL = 41.43
# => Average : Train PPL = 31.78 ± 0.09 | Val PPL = 41.32 ± 0.12

# Config 3 (pre_norm=True, post_norm=True, qk_norm=True)
# SEED 12 : Best Train PPL = 30.91 | Best Val PPL = 41.01
# SEED 42 : Best Train PPL = 30.79 | Best Val PPL = 40.94
# SEED 100 : Best Train PPL = 30.85 | Best Val PPL = 41.22
# => Average : Train PPL = 30.85 ± 0.06 | Val PPL = 41.06 ± 0.15
